In [7]:
from Environment import *
from DDPG import *
from NN_Module import *
import os
from config import *
import sys

import torch
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from numpy import linalg as LA

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.colors as pc
from pandapower.plotting.plotly import pf_res_plotly

from loguru import logger

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
print(f"Using device: {device}")

### a simple logger
logger.remove()
logger.add(sys.stderr, level='INFO')


Using device: cpu


6

In [3]:
env_seed = 7        #10-h  5-h 0-l 1-h 2-l 3-l 4l 7h 8h 9l
episode_num = 3000   # the total test episode
step_num = 200      # the longest test step

### create testing environment
injection_bus = np.array([18, 21, 30, 45, 53])-1
pp_net = create_56bus()
env = VoltageCtrl_Env(pp_net, injection_bus)
state, topology, senario = env.reset_topo(seed=env_seed)
# topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
# pf_res_plotly(pp_net);

d:\Code\Python\Flexible_Voltage_Control\.venv\Lib\site-packages\pandapower\converter\pypower\from_ppc.py:334: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  branch_lookup.loc[is_trafo, "element"] = idx_trafo


### Some Plot Function

In [4]:
def moving_average(a, n=3):
    # Padding the array to maintain the length after convolution.
    pad = np.pad(a, (n//2, n-1-n//2), mode='edge')
    ret = np.convolve(pad, np.ones(n), mode='valid') / n
    return ret

# plot policy
def policy_plotly(policy_net, topology):
    """
    用 Plotly 绘制各母线的策略曲线，每个子图显示一个母线的 RLC-FT 策略与基线（Linear）策略比较，
    """
    default_colors = pc.qualitative.Plotly  # Plotly 默认颜色序列
    color_linear = default_colors[0]
    color_rlc = default_colors[1]
    fig = make_subplots(rows=1, cols=5,
                        subplot_titles=['Bus 18', 'Bus 21', 'Bus 30', 'Bus 45', 'Bus 53'])
    N = 400
    for i in range(5):
        baseline_vals = []
        policy_vals = []
        for j in range(N):
            # 计算基线控制值：baseline = max(state-1.05, 0) - max(0.95-state, 0)
            state_val = 0.80 + 0.001 * j
            base = np.maximum(state_val - 1.05, 0) - np.maximum(0.95 - state_val, 0)
            baseline_vals.append(-base)  # 取负值
            state_tensor = torch.tensor([[state_val]])
            action_tensor = policy_net[i](state_tensor, topology)
            policy_vals.append(float(-action_tensor.detach().cpu().numpy()[0]))

        baseline_vals = np.array(baseline_vals)
        policy_vals_smoothed = moving_average(np.array(policy_vals), n=20)
        baseline_vals_scaled = 5 * baseline_vals
        
        x_vals = np.linspace(10, 14, N)
        
        # 仅在第一列显示图例，其余子图同组 trace 设为不显示图例
        showlegend = True if i == 0 else False

        fig.add_trace(go.Scatter(
            x=x_vals,
            y=baseline_vals_scaled,
            mode='lines',
            name='Linear',
            legendgroup='Linear',
            showlegend=showlegend,
            line=dict(dash='dash', color=color_linear)
        ), row=1, col=i+1)

        fig.add_trace(go.Scatter(
            x=x_vals,
            y=policy_vals_smoothed,
            mode='lines',
            name='RLC-FT',
            legendgroup='RLC-FT',
            showlegend=showlegend,
            line=dict(color=color_rlc)
        ), row=1, col=i+1)

    # 保证仅在第一个子图显示y轴标题，第三个子图显示x轴标题
    fig.update_yaxes(title_text="Q (MVar)", row=1, col=1)
    fig.update_xaxes(title=dict(text="Voltage (kV)", standoff=25), row=1, col=3)
    fig.update_layout(
        width=1400,
        height=500,
        showlegend=True,
        font=dict(size=16),
        xaxis=dict(
            tickfont=dict(size=12),
            showline=True,
            mirror=True,
            showgrid=True,
        ),
        yaxis=dict(
            tickfont=dict(size=12),
            showline=True,
            mirror=True,
            showgrid=True,
        ),
    )
    
    output_path = os.path.join(Config.data_path, 'images', '56bus', 'policy_plot.pdf')
    import plotly.io as pio
    pio.kaleido.scope.mathjax = None
    fig.write_image(output_path)
    fig.show()


def safe_net_plotly(safe_net):
    """
    用 Plotly 绘制 safe network 策略曲线，每个子图显示一个母线的 Stable-DDPG 与 Linear 比较
    """
    default_colors = pc.qualitative.Plotly  # Plotly 默认颜色序列
    color_linear = default_colors[0]
    color_safe = default_colors[1]
    fig = make_subplots(rows=1, cols=5,
                        subplot_titles=['Bus 18', 'Bus 21', 'Bus 30', 'Bus 45', 'Bus 53'])
    N = 400
    for i in range(len(safe_net)):
        baseline_vals = []
        safe_vals = []
        for j in range(N):
            state_val = 0.80 + 0.001 * j
            base = np.maximum(state_val - 1.05, 0) - np.maximum(0.95 - state_val, 0)
            baseline_vals.append(-base)
            # safe_net[i].get_action 接受列表输入，返回单个数值
            action = safe_net[i].get_action([state_val])
            safe_vals.append(-float(action))
        baseline_vals = np.array(baseline_vals)
        baseline_vals_scaled = 2 * baseline_vals
        x_vals = np.linspace(10, 14, N)
        # 仅在第一列显示图例，其余子图同组 trace 设为不显示图例
        showlegend = True if i == 0 else False

        fig.add_trace(go.Scatter(
            x=x_vals,
            y=baseline_vals_scaled,
            mode='lines',
            name='Linear',
            showlegend=showlegend,
            line=dict(dash='dash', color=color_linear)
        ), row=1, col=i+1)

        fig.add_trace(go.Scatter(
            x=x_vals,
            y=safe_vals,
            mode='lines',
            name='Safe-DDPG',
            showlegend=showlegend,
            line=dict(color=color_safe)
        ), row=1, col=i+1)

    # 保证仅在第一个子图显示y轴标题，第三个子图显示x轴标题
    fig.update_yaxes(title_text="Q (MVar)", row=1, col=1)
    fig.update_xaxes(title=dict(text="Voltage (kV)", standoff=25), row=1, col=3)
    fig.update_layout(
        width=1400,
        height=500,
        showlegend=True,
        xaxis=dict(
            showline=True,
            mirror=True,
            showgrid=True,
        ),
        yaxis=dict(
            showline=True,
            mirror=True,
            showgrid=True,
        ),
    )

    fig.show()


def x_policy_plotly(policy_net):
    """
    用 Plotly 绘制不同拓扑下的 RLC-FT 策略曲线，所有情形绘制在单个图中
    """
    import plotly.graph_objects as go
    fig = go.Figure()
    N = 400
    for i in range(5):
        policy_vals = []
        # 对于每个拓扑情形，通过 reset_topo 获得对应拓扑设定
        state, topo, senario = env.reset_topo(seed=i)
        topo_tensor = torch.cuda.FloatTensor(topo).unsqueeze(0)
        for j in range(N):
            state_tensor = torch.tensor([[0.80 + 0.001 * j]])
            action_tensor = policy_net[2](state_tensor, topo_tensor)
            policy_vals.append(float(-action_tensor.detach().cpu().numpy()[0]))
        policy_vals_smoothed = moving_average(np.array(policy_vals), n=20)
        x_vals = np.linspace(10, 14, N)
        fig.add_trace(go.Scatter(x=x_vals, y=policy_vals_smoothed,
                                 mode='lines',
                                 name=f'Topology {i}'))
    fig.update_layout(
        font=dict(size=16),
        width=700,
        height=500,
        # margin=dict(l=30, r=30, t=30, b=30),   # Uncomment to adjust margins
        margin=dict(r=30,t=30,b=60),
        xaxis_title='Voltage (kV)',
        yaxis_title='Q (MVar)',
        xaxis=dict(
            showgrid=True,
            tickfont=dict(size=12),
        ),
        yaxis=dict(
            tickfont=dict(size=12),
            showgrid=True,
            zeroline=True,
            zerolinecolor='lightgray'
        ),
        legend=dict(
            x=1,
            y=1,
            xanchor='right',
            yanchor='top',
            bgcolor='rgba(255,255,255,1.0)'
        ),
    )

    fig.show()

### Load model

In [5]:
agent_num = 5
agent_policy_net = []
safe_agent_net = []

### load nn model parameter from saved model 
for i in range(agent_num):
    topology_net = TopologyNet(topology_dim=55, output_dim=1, hidden_dim=256)
    policy_net = FlexiblePolicyNet(env=env, topology_net=topology_net, obs_dim=1, action_dim=1, hidden_dim=2048).to(device)
    agent_policy_net.append(policy_net)

for i in range(agent_num):
    policy_net = SafePolicyNetwork(env=env, obs_dim=1, action_dim=1, hidden_dim=100).to(device)
    safe_agent_net.append(policy_net)

for i in range(agent_num):
    #value_net_dict = torch.load(f'check_points/value_net/2023-06-19/Step_200_Seed_12_a{i}.pth')
    #policy_net_dict = torch.load(f'check_points/policy_net/2023-07-05/Step_300_Seed_45_a{i}.pth')
    # policy_net_dict = torch.load(f'check_points/policy_net/2023-08-15/Step_900_Seed_33_a{i}.pth')
    #policy_net_dict = torch.load(f'check_points/policy_net/2023-09-21/Step_900_Seed_10_a{i}.pth')
    # policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2025-02-18/Step_600_Seed_12_a{i}.pth'))
    policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2025-02-18/Step_500_Seed_4_a{i}.pth'))

    agent_policy_net[i].load_state_dict(policy_net_dict)

for i in range(agent_num):
    #value_net_dict = torch.load(f'D:/Code/Python/StableRL_VoltageCtrl-main/saved_models/2023-06-19/SafeDDPG_value_Step_200_a{i}.pth')
    policy_net_dict = torch.load(f'D:/Code/Python/StableRL_VoltageCtrl-main/saved_models/stable_ddpg/policy_net_checkpoint_a{i}.pth')
    safe_agent_net[i].load_state_dict(policy_net_dict)

RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

In [15]:
policy_plotly(agent_policy_net, topology)
# safe_net_plotly(safe_agent_net)

C:\Users\wdyao\AppData\Local\Temp\ipykernel_36048\3687654066.py:28: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

C:\Users\wdyao\AppData\Local\Temp\ipykernel_36048\3687654066.py:83: DeprecationWarning:


Use of plotly.io.kaleido.scope.mathjax is deprecated and support will be removed after September 2025.
Please use plotly.io.defaults.mathjax instead.




### Flexible NN Contoller

In [16]:
### test our controller
voltage = []
q = []
cost = []
success_list = []
fail_list = []
entire_list = []
control_cost = []
reward_list = []
object_cost = []
voltage_trajs = []

for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    cost = []
    abnormal_stop = False

    voltage_episode = []   # stores full voltage vectors for this episode

    for step in range(step_num):
        action = []
        for i in range(agent_num):
            action_agent = agent_policy_net[i](torch.cuda.FloatTensor(state[i].reshape(1,)).unsqueeze(0), topology)
            action_agent = 0.7 * action_agent.detach().cpu().numpy()[0] #0.7
            action.append(action_agent)

        action = last_action - np.asarray(action)
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            fail_list.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            fail_list.append((episode,step))
            abnormal_stop = True
            break
        if done:
            success_list.append((episode,step))
            logger.success('episode {} stable at {} steps',success_list[-1][0], success_list[-1][1])
            break

        voltage.append(state)
        voltage_episode.append(state.copy())  # record full state vector

        q.append(action)

        state = next_state
        
        episode_reward += reward
        
        cost.append(-reward)
        
        episode_control += LA.norm(action, 2)

    reward_list.append(episode_reward)
    control_cost.append(episode_control)
    object_cost.append(np.sum(cost))
    if len(voltage_episode) > 0 and senario == 0:
        voltage_trajs.append(np.vstack(voltage_episode))

    if (not done) and (abnormal_stop == False):
        entire_list.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

logger.info('total success epsisode is {}', len(success_list))
logger.info('total fail episode is {}', len(fail_list))
logger.info('number of finished at entire episode is {}', len(entire_list))

2025-09-03 17:04:47.678 | SUCCESS  | __main__:<module>:51 - episode 0 stable at 7 steps
2025-09-03 17:04:47.849 | SUCCESS  | __main__:<module>:51 - episode 1 stable at 2 steps
2025-09-03 17:04:48.274 | SUCCESS  | __main__:<module>:51 - episode 2 stable at 7 steps
2025-09-03 17:04:48.537 | SUCCESS  | __main__:<module>:51 - episode 3 stable at 5 steps
2025-09-03 17:04:48.795 | SUCCESS  | __main__:<module>:51 - episode 4 stable at 5 steps
2025-09-03 17:04:49.350 | SUCCESS  | __main__:<module>:51 - episode 5 stable at 13 steps
2025-09-03 17:04:49.460 | SUCCESS  | __main__:<module>:51 - episode 6 stable at 1 steps
2025-09-03 17:04:50.783 | SUCCESS  | __main__:<module>:51 - episode 7 stable at 33 steps
2025-09-03 17:04:51.481 | SUCCESS  | __main__:<module>:51 - episode 8 stable at 17 steps
2025-09-03 17:04:51.966 | SUCCESS  | __main__:<module>:51 - episode 9 stable at 11 steps
2025-09-03 17:04:52.489 | SUCCESS  | __main__:<module>:51 - episode 10 stable at 12 steps
2025-09-03 17:04:52.768 | 

In [17]:
success_list = np.array(success_list)
print('average recovery step is:')
print(np.mean(success_list[:,1]))
print(np.std(success_list[:,1]))
print('average reactive power cost is:')
print(np.mean(control_cost))
print(np.std(control_cost))
print('the total cost is:')
# print(object_cost)
print(np.mean(object_cost))
print(np.std(object_cost))

import plotly.express as px

# ---------- 3. plotting ----------
bus_idx = 2                                   # choose bus (0-based)
max_len  = max(a.shape[0] for a in voltage_trajs)

# build (episodes × max_len) matrix with NaN padding
traj_mat = np.full((len(voltage_trajs), max_len), np.nan)
for i, ep in enumerate(voltage_trajs):
    traj_mat[i, :ep.shape[0]] = ep[:, bus_idx]

mean_traj = np.nanmean(traj_mat, axis=0)
min_traj  = np.nanmin(traj_mat, axis=0)
max_traj  = np.nanmax(traj_mat, axis=0)

# pick one color from Plotly's qualitative palette
base_color = px.colors.qualitative.Plotly[0]  # '#1f77b4' (blue)

def hex_to_rgba(hex_color, alpha):
    """#1f77b4 -> rgba(31,119,180,alpha)"""
    h = hex_color.lstrip('#')
    r, g, b = tuple(int(h[i:i+2], 16) for i in (0, 2, 4))
    return f'rgba({r},{g},{b},{alpha})'

fill_color = hex_to_rgba(base_color, 0.25)    # translucent blue

fig = go.Figure()

# 1) lower bound (invisible line – starting edge of the band)
fig.add_trace(
    go.Scatter(
        x=np.arange(max_len),
        y=min_traj,
        mode="lines",
        line=dict(width=0),
        showlegend=False,
        hoverinfo="skip",
        name="min",
    )
)

# 2) upper bound with 'tonexty' fill – creates the shaded band
fig.add_trace(
    go.Scatter(
        x=np.arange(max_len),
        y=max_traj,
        mode="lines",
        line=dict(width=0),
        fill="tonexty",
        fillcolor=fill_color,
        showlegend=False,
        hoverinfo="skip",
        name="max",
    )
)

# 3) mean curve on top, same hue but solid & thicker
fig.add_trace(
    go.Scatter(
        x=np.arange(max_len),
        y=mean_traj,
        mode="lines",
        name=f"Mean Voltage (Bus {bus_idx + 1})",
        line=dict(color=base_color, width=3),
    )
)

fig.update_layout(
    title=f"Voltage Trajectories on Bus {bus_idx + 1}",
    xaxis_title="Step",
    yaxis_title="Voltage (p.u.)",
    template="plotly_white",
)

fig.show()

average recovery step is:
8.043333333333333
8.598146441077997
average reactive power cost is:
33.7250230752951
66.64942575285309
the total cost is:
137.3305099794837
159.69847886817345


In [18]:
### test our controller without topology change
voltage_ = []
q_ = []
cost_ = []
success_list_ = []
fail_list_ = []
entire_list_ = []
control_cost_ = []
reward_list_ = []
object_cost_list_ = []
voltage_trajs_ = []

def apply_hybrid_error(true_topology: torch.Tensor, error_percentage: float) -> torch.Tensor:
    """
    Applies a hybrid error model to a topology tensor.

    For a specified percentage of connected lines, this function will either
    set the line's susceptance to zero (simulating disconnection) or perturb
    it with multiplicative Gaussian noise (simulating parameter corruption).

    Args:
        true_topology: The original, correct topology tensor.
        error_percentage: The fraction of connected lines to apply an error to (e.g., 0.3 for 30%).

    Returns:
        A new topology tensor with the errors applied.
    """
    # Create a copy to modify
    believed_topology = true_topology.clone()

    # 1. Find the indices of all connected lines (non-zero elements)
    connected_indices = torch.nonzero(true_topology)
    num_connected_lines = connected_indices.size(0)

    # 2. Determine how many lines to apply an error to
    num_lines_to_corrupt = int(num_connected_lines * error_percentage)

    # 3. Randomly select the indices of the lines that will have an error
    corruption_indices_perm = torch.randperm(num_connected_lines)
    indices_to_corrupt = connected_indices[corruption_indices_perm[:num_lines_to_corrupt]]

    # 4. Apply the hybrid error to each selected line
    for idx_tuple in indices_to_corrupt:
        # This index tuple is something like (batch, channel, row, col)
        # We need to convert it to a simple tuple for indexing
        idx = tuple(idx_tuple.tolist())

        # 50% chance to simulate disconnection (set to zero)
        if torch.rand(1).item() < 0.8:
            believed_topology[idx] = 0.0
        # 50% chance to simulate parameter corruption
        else:
            original_value = believed_topology[idx]
            # Get a random perturbation factor from N(mean=1.0, std=0.5)
            perturb_factor = torch.normal(mean=1.0, std=1.0, size=(1,)).item()
            # Apply perturbation, ensuring the value is not negative
            believed_topology[idx] = max(0.0, original_value * perturb_factor)

    return believed_topology

for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
    believed_topology = apply_hybrid_error(topology, error_percentage=0.5)

    # print(f'Topology for episode {episode}: {believed_topology}')

    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    cost_ = []
    abnormal_stop = False
    voltage_episode = []   # stores full voltage vectors for this episode

    for step in range(step_num):
        action = []
        for i in range(agent_num):
            action_agent = agent_policy_net[i](torch.cuda.FloatTensor(state[i].reshape(1,)).unsqueeze(0), believed_topology)
            action_agent = 0.6* action_agent.detach().cpu().numpy()[0]
            action.append(action_agent)

        action = last_action - np.asarray(action)
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            fail_list_.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            fail_list_.append((episode,step))
            abnormal_stop = True
            break
        if done:
            success_list_.append((episode,step))
            logger.success('episode {} stable at {} steps',success_list_[-1][0], success_list_[-1][1])
            break

        voltage_.append(state)
        voltage_episode.append(state.copy())      # record full state vector

        q_.append(action)

        state = next_state
        
        episode_reward += reward
        
        cost_.append(-reward)
        
        episode_control += LA.norm(action, 2)

    reward_list_.append(episode_reward)
    control_cost_.append(episode_control)
    object_cost_list_.append(np.sum(cost_))
    if len(voltage_episode) > 0 and senario == 0:
        voltage_trajs_.append(np.vstack(voltage_episode))

    if (not done) and (abnormal_stop == False):
        entire_list_.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

logger.info('total success epsisode is {}', len(success_list_))
logger.info('total fail episode is {}', len(fail_list_))
logger.info('number of finished at entire episode is {}', len(entire_list_))

2025-09-03 17:06:36.598 | SUCCESS  | __main__:<module>:102 - episode 0 stable at 8 steps
2025-09-03 17:06:36.786 | SUCCESS  | __main__:<module>:102 - episode 1 stable at 3 steps
2025-09-03 17:06:37.225 | SUCCESS  | __main__:<module>:102 - episode 2 stable at 10 steps
2025-09-03 17:06:37.575 | SUCCESS  | __main__:<module>:102 - episode 3 stable at 7 steps
2025-09-03 17:06:37.935 | SUCCESS  | __main__:<module>:102 - episode 4 stable at 8 steps
2025-09-03 17:06:38.535 | SUCCESS  | __main__:<module>:102 - episode 5 stable at 14 steps
2025-09-03 17:06:38.686 | SUCCESS  | __main__:<module>:102 - episode 6 stable at 2 steps
2025-09-03 17:06:40.146 | SUCCESS  | __main__:<module>:102 - episode 7 stable at 38 steps
2025-09-03 17:06:41.146 | SUCCESS  | __main__:<module>:102 - episode 8 stable at 25 steps
2025-09-03 17:06:41.744 | SUCCESS  | __main__:<module>:102 - episode 9 stable at 15 steps
2025-09-03 17:06:42.371 | SUCCESS  | __main__:<module>:102 - episode 10 stable at 16 steps
2025-09-03 17:

In [19]:
success_list_ = np.array(success_list_)
print('average recovery step is:')
print(np.mean(success_list_[:,1]))
print(np.std(success_list_[:,1]))
print('average reactive power cost is:')
print(np.mean(control_cost_))
print(np.std(control_cost_))
print('the total cost is:')
print(np.mean(object_cost_list_))
print(np.std(object_cost_list_))

import os
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

# ------------------------------------------------------------
# Nature-style figure configuration
# ------------------------------------------------------------
NATURE_CONFIG = {
    "width": 1800,
    "height": 900,
    "font_base": 28,
    "font_title": 32,
    "font_axis": 24,
    "font_legend": 24,
    "dpi": 300,          # only used indirectly through scale
}

# ------------------------------------------------------------
# helper functions
# ------------------------------------------------------------
def build_stats(trajs, bus_idx, max_len):
    mat = np.full((len(trajs), max_len), np.nan)
    for i, ep in enumerate(trajs):
        mat[i, :ep.shape[0]] = ep[:, bus_idx]
    return {
        "mean": np.nanmean(mat, axis=0),
        "min":  np.nanmin(mat, axis=0),
        "max":  np.nanmax(mat, axis=0)
    }

def hex_to_rgba(hex_color, alpha):
    h = hex_color.lstrip('#')
    r, g, b = (int(h[i:i+2], 16) for i in (0, 2, 4))
    return f'rgba({r},{g},{b},{alpha})'

# ------------------------------------------------------------
# choose bus & palette
# ------------------------------------------------------------
bus_idx = 2               # Bus-30
color_perfect = "#1b9e77" # teal   – perfect topology
color_error   = "#d95f02" # orange – 50 % error

max_len = max(
    max(a.shape[0] for a in voltage_trajs),
    max(a.shape[0] for a in voltage_trajs_)
)

stats_perfect = build_stats(voltage_trajs,  bus_idx, max_len)
stats_error   = build_stats(voltage_trajs_, bus_idx, max_len)

# ------------------------------------------------------------
# build figure
# ------------------------------------------------------------
fig = go.Figure()

# ---- shaded band : perfect topology ----
fig.add_trace(go.Scatter(
    x=np.arange(max_len), y=stats_perfect["min"],
    mode="lines", line=dict(width=0),
    showlegend=False, hoverinfo="skip",
    legendgroup="perfect"
))
fig.add_trace(go.Scatter(
    x=np.arange(max_len), y=stats_perfect["max"],
    mode="lines", line=dict(width=0),
    fill="tonexty", fillcolor=hex_to_rgba(color_perfect, 0.25),
    showlegend=False, hoverinfo="skip",
    legendgroup="perfect"
))

# ---- shaded band : 50 % error ----
fig.add_trace(go.Scatter(
    x=np.arange(max_len), y=stats_error["min"],
    mode="lines", line=dict(width=0),
    showlegend=False, hoverinfo="skip",
    legendgroup="error"
))
fig.add_trace(go.Scatter(
    x=np.arange(max_len), y=stats_error["max"],
    mode="lines", line=dict(width=0),
    fill="tonexty", fillcolor=hex_to_rgba(color_error, 0.25),
    showlegend=False, hoverinfo="skip",
    legendgroup="error"
))

# ---- mean curves ----
fig.add_trace(go.Scatter(
    x=np.arange(max_len), y=stats_perfect["mean"],
    mode="lines",
    name="Perfect information (mean)",
    line=dict(color=color_perfect, width=5),
    legendgroup="perfect"
))
fig.add_trace(go.Scatter(
    x=np.arange(max_len), y=stats_error["mean"],
    mode="lines",
    name="50 % error information (mean)",
    line=dict(color=color_error, width=5),
    legendgroup="error"
))

# ---- safety threshold ----
fig.add_trace(go.Scatter(
    x=[0, max_len - 1], y=[0.95, 0.95],
    mode="lines", name="Safety limit 0.95 p.u.",
    line=dict(color="black", dash="dash"),
    hoverinfo="skip"
))

# ------------------------------------------------------------
# layout: Nature style
# ------------------------------------------------------------
fig.update_layout(
    width=NATURE_CONFIG["width"],
    height=NATURE_CONFIG["height"],
    template="plotly_white",
    title=dict(
        text=f"Average Voltage on Bus 30: perfect vs 50 % error topology information",
        font=dict(size=NATURE_CONFIG["font_title"])
    ),
    font=dict(size=NATURE_CONFIG["font_base"]),
    legend=dict(
        x=0.99, y=0.99, xanchor="right", yanchor="top",
        font=dict(size=NATURE_CONFIG["font_legend"])
    ),
    xaxis=dict(
        title="Step",
        titlefont=dict(size=NATURE_CONFIG["font_axis"]),
        tickfont=dict(size=NATURE_CONFIG["font_axis"]),
        range=[0, 18],
    ),
    yaxis=dict(
        title="Voltage (p.u.)",
        titlefont=dict(size=NATURE_CONFIG["font_axis"]),
        tickfont=dict(size=NATURE_CONFIG["font_axis"])
    )
)

fig.show()

# ------------------------------------------------------------
# export high-resolution images
# ------------------------------------------------------------
pio.kaleido.scope.mathjax = None  # speed-up export

output_dir = os.path.join(Config.data_path, "images", "56bus")
os.makedirs(output_dir, exist_ok=True)

pdf_path = os.path.join(output_dir, "voltage_plot.pdf")
png_path = os.path.join(output_dir, "voltage_plot.png")

fig.write_image(pdf_path,  width=NATURE_CONFIG["width"],
                           height=NATURE_CONFIG["height"],
                           scale=2)   # 2 × 1800 px ≈ 300 dpi
fig.write_image(png_path,  width=NATURE_CONFIG["width"],
                           height=NATURE_CONFIG["height"],
                           scale=2)

average recovery step is:
11.71
11.740211525635587
average reactive power cost is:
47.28938579326559
91.2251223781093
the total cost is:
195.12178375309324
217.68224245149455


C:\Users\wdyao\AppData\Local\Temp\ipykernel_36048\1156879876.py:38: RuntimeWarning:

Mean of empty slice

C:\Users\wdyao\AppData\Local\Temp\ipykernel_36048\1156879876.py:39: RuntimeWarning:

All-NaN slice encountered

C:\Users\wdyao\AppData\Local\Temp\ipykernel_36048\1156879876.py:40: RuntimeWarning:

All-NaN slice encountered



ValueError: Invalid property specified for object of type plotly.graph_objs.layout.XAxis: 'titlefont'

Did you mean "tickfont"?

    Valid properties:
        anchor
            If set to an opposite-letter axis id (e.g. `x2`, `y`),
            this axis is bound to the corresponding opposite-letter
            axis. If set to "free", this axis' position is
            determined by `position`.
        automargin
            Determines whether long tick labels automatically grow
            the figure margins.
        autorange
            Determines whether or not the range of this axis is
            computed in relation to the input data. See `rangemode`
            for more info. If `range` is provided and it has a
            value for both the lower and upper bound, `autorange`
            is set to False. Using "min" applies autorange only to
            set the minimum. Using "max" applies autorange only to
            set the maximum. Using *min reversed* applies autorange
   

In [ ]:
def apply_hybrid_error(true_topology: torch.Tensor, error_percentage: float) -> torch.Tensor:
    # Create a copy to modify
    believed_topology = true_topology.clone()

    # 1. Find the indices of all connected lines (non-zero elements)
    connected_indices = torch.nonzero(true_topology)
    num_connected_lines = connected_indices.size(0)

    # 2. Determine how many lines to apply an error to
    num_lines_to_corrupt = int(num_connected_lines * error_percentage)

    # 3. Randomly select the indices of the lines that will have an error
    corruption_indices_perm = torch.randperm(num_connected_lines)
    indices_to_corrupt = connected_indices[corruption_indices_perm[:num_lines_to_corrupt]]

    # 4. Apply the hybrid error to each selected line
    for idx_tuple in indices_to_corrupt:
        # This index tuple is something like (batch, channel, row, col)
        # We need to convert it to a simple tuple for indexing
        idx = tuple(idx_tuple.tolist())

        # 50% chance to simulate disconnection (set to zero)
        if torch.rand(1).item() < 0.5:
            believed_topology[idx] = 0.0
        # 50% chance to simulate parameter corruption
        else:
            original_value = believed_topology[idx]
            # Get a random perturbation factor from N(mean=1.0, std=0.5)
            perturb_factor = torch.normal(mean=1.0, std=0.5, size=(1,)).item()
            # Apply perturbation, ensuring the value is not negative
            believed_topology[idx] = max(0.0, original_value * perturb_factor)

    return believed_topology

# ============================================================================ #
#                         H Y P E R   P A R A M E T E R S                      #
# ---------------------------------------------------------------------------- #
loss_prob_arr     = 0.5          # scalar or array → packet-loss probability
min_delay_arr     = 0            # make delay >= 3 so most episodes feel it
max_delay_arr     = 3
noise_prob_arr    = 0.5          # chance a delivered packet is corrupted
# ============================================================================ #

class CommChannel:
    """Packet-loss / delay / corruption plus an initial black-out window."""
    def __init__(self, loss_prob, min_delay, max_delay, noise_prob):
        self.loss_prob   = loss_prob
        self.min_delay   = min_delay
        self.max_delay   = max_delay
        self.noise_prob  = noise_prob
        self.queue       = []               # [(arrival_step, topo_tensor), …]
        self.current     = None             # last successfully received topology

    def step(self, true_topology: torch.Tensor, t: int) -> torch.Tensor:
        # 1) deliver everything whose arrival time is now
        while self.queue and self.queue[0][0] == t:
            _, topo = self.queue.pop(0)
            self.current = topo.clone()

        # 2) send a NEW packet 
        if np.random.rand() > self.loss_prob:
            delay  = np.random.randint(self.min_delay, self.max_delay + 1)
            packet = true_topology.clone()
            if np.random.rand() < self.noise_prob:
                packet = apply_hybrid_error(packet, error_percentage=0.5)
            self.queue.append((t + delay, packet))
            self.queue.sort(key=lambda x: x[0])

        # 3) return whatever we currently know (may be None)
        return self.current
# --------------------------------------------------------------------------- #


# ============================== result buffers ============================== #
voltage_loss          = []
q_loss                = []
reward_list_loss      = []
control_cost_loss     = []
object_cost_list_loss = []
success_list_loss     = []
fail_list_loss        = []
entire_list_loss      = []
voltage_trajs_loss    = []
# =========================================================================== #

for episode in range(episode_num):
    state, true_topology, scenario = env.reset_topo(seed=episode)
    true_topology = torch.cuda.FloatTensor(true_topology).unsqueeze(0)

    # ------------------- build one channel per agent -----------------------
    channels = []
    for i in range(agent_num):
        ch = CommChannel(
            loss_prob   = loss_prob_arr  if np.ndim(loss_prob_arr)  == 0 else loss_prob_arr[i],
            min_delay   = min_delay_arr  if np.ndim(min_delay_arr)  == 0 else min_delay_arr[i],
            max_delay   = max_delay_arr  if np.ndim(max_delay_arr)  == 0 else max_delay_arr[i],
            noise_prob  = noise_prob_arr if np.ndim(noise_prob_arr) == 0 else noise_prob_arr[i],
        )

        # ----------- bootstrap topology knowledge at t = 0 ----------------
        if np.random.rand() > loss_prob_arr:
            if np.random.rand() < noise_prob_arr:
                # apply hybrid error to the true topology
                ch.current = apply_hybrid_error(true_topology.clone(), 0.5)
            else:
                # no error, just copy the true topology
                ch.current = true_topology.clone()
        # "none": leave ch.current = None
        channels.append(ch)
    # ----------------------------------------------------------------------

    last_action      = np.zeros((agent_num, 1))
    episode_reward   = 0.0
    episode_control  = 0.0
    step_cost_list   = []
    abnormal_stop    = False
    voltage_episode  = []

    for step in range(step_num):

        # -------- per-agent, possibly None, believed topologies -----------
        believed_topos = [ch.step(true_topology, step) for ch in channels]

        # When an agent still has no information (None), you can choose:
        #   • Skip its control (keep previous action),
        #   • Feed a zero tensor,
        #   • Feed some default known topology.
        # Here we fall back to a zero tensor with the correct shape.
        for k, topo in enumerate(believed_topos):
            if topo is None:
                believed_topos[k] = torch.zeros_like(true_topology)

        # -------------------- compute agent actions -----------------------
        actions = []
        for i in range(agent_num):
            a_i = agent_policy_net[i](
                torch.cuda.FloatTensor(state[i].reshape(1,)).unsqueeze(0),
                believed_topos[i]
            )
            a_i = 0.6 * a_i.detach().cpu().numpy()[0]
            actions.append(a_i)
        actions = last_action - np.asarray(actions)
        last_action = np.copy(actions)
        # -----------------------------------------------------------------

        # ------------------------ environment step ------------------------
        try:
            next_state, reward, done = env.step(actions)
        except pp.powerflow.LoadflowNotConverged:
            logger.error('power flow not converge at episode {} step {}', episode, step)
            fail_list_loss.append((episode, step))
            abnormal_stop = True
            break

        if np.min(next_state) < 0.75 or np.max(next_state) > 1.25:
            logger.warning('episode {} step {} voltage violation', episode, step)
            fail_list_loss.append((episode, step))
            abnormal_stop = True
            break

        if done:
            success_list_loss.append((episode, step))
            logger.success('episode {} stable at {} steps', episode, step)
            break
        # -----------------------------------------------------------------

        # --------------------------- bookkeeping --------------------------
        voltage_loss.append(state)
        voltage_episode.append(state.copy())
        q_loss.append(actions)

        state = next_state
        episode_reward  += reward
        step_cost_list.append(-reward)
        episode_control += np.linalg.norm(actions, 2)
        # -----------------------------------------------------------------

    # ================= episode-level bookkeeping ==========================
    reward_list_loss.append(episode_reward)
    control_cost_loss.append(episode_control)
    object_cost_list_loss.append(np.sum(step_cost_list))
    if voltage_episode and scenario == 0:
        voltage_trajs_loss.append(np.vstack(voltage_episode))

    if (not done) and (not abnormal_stop):
        entire_list_loss.append(episode)
        logger.info('Episode {} finished with full horizon', episode)

# =========================== final statistics ============================== #
logger.info('total success episodes (lossy): {}', len(success_list_loss))
logger.info('total fail episodes    (lossy): {}', len(fail_list_loss))
logger.info('episodes ran full horizon (lossy): {}', len(entire_list_loss))

2025-06-28 18:38:58.676 | SUCCESS  | __main__:<module>:163 - episode 0 stable at 8 steps
2025-06-28 18:38:58.916 | SUCCESS  | __main__:<module>:163 - episode 1 stable at 2 steps
2025-06-28 18:38:59.557 | SUCCESS  | __main__:<module>:163 - episode 2 stable at 10 steps
2025-06-28 18:38:59.897 | SUCCESS  | __main__:<module>:163 - episode 3 stable at 6 steps
2025-06-28 18:39:00.320 | SUCCESS  | __main__:<module>:163 - episode 4 stable at 8 steps
2025-06-28 18:39:01.251 | SUCCESS  | __main__:<module>:163 - episode 5 stable at 17 steps
2025-06-28 18:39:01.579 | SUCCESS  | __main__:<module>:163 - episode 6 stable at 4 steps
2025-06-28 18:39:03.946 | SUCCESS  | __main__:<module>:163 - episode 7 stable at 46 steps
2025-06-28 18:39:04.900 | SUCCESS  | __main__:<module>:163 - episode 8 stable at 20 steps
2025-06-28 18:39:06.778 | SUCCESS  | __main__:<module>:163 - episode 9 stable at 38 steps
2025-06-28 18:39:07.582 | SUCCESS  | __main__:<module>:163 - episode 10 stable at 16 steps
2025-06-28 18:

In [ ]:
success_list_loss = np.array(success_list_loss)
print('average recovery step is:')
print(np.mean(success_list_loss[:,1]))
print(np.std(success_list_loss[:,1]))
print('average reactive power cost is:')
print(np.mean(control_cost_loss))
print(np.std(control_cost_loss))
print('the total cost is:')
print(np.mean(object_cost_list_loss))
print(np.std(object_cost_list_loss))

import os
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

# ------------------------------------------------------------
# Nature-style figure configuration
# ------------------------------------------------------------
NATURE_CONFIG = {
    "width": 1800,
    "height": 900,
    "font_base": 28,
    "font_title": 32,
    "font_axis": 24,
    "font_legend": 24,
    "dpi": 300,          # only used indirectly through scale
}

# ------------------------------------------------------------
# helper functions
# ------------------------------------------------------------
def build_stats(trajs, bus_idx, max_len):
    mat = np.full((len(trajs), max_len), np.nan)
    for i, ep in enumerate(trajs):
        mat[i, :ep.shape[0]] = ep[:, bus_idx]
    return {
        "mean": np.nanmean(mat, axis=0),
        "min":  np.nanmin(mat, axis=0),
        "max":  np.nanmax(mat, axis=0)
    }

def hex_to_rgba(hex_color, alpha):
    h = hex_color.lstrip('#')
    r, g, b = (int(h[i:i+2], 16) for i in (0, 2, 4))
    return f'rgba({r},{g},{b},{alpha})'

# ------------------------------------------------------------
# choose bus & palette
# ------------------------------------------------------------
bus_idx = 2               # Bus-30
color_perfect = "#1b9e77" # teal   – 0% loss
color_error   = "#d95f02" # orange – 50 % loss

max_len = max(
    max(a.shape[0] for a in voltage_trajs),
    max(a.shape[0] for a in voltage_trajs_loss)
)

stats_perfect = build_stats(voltage_trajs,  bus_idx, max_len)
stats_error   = build_stats(voltage_trajs_loss, bus_idx, max_len)

# ------------------------------------------------------------
# build figure
# ------------------------------------------------------------
fig = go.Figure()

# ---- shaded band : perfect topology ----
fig.add_trace(go.Scatter(
    x=np.arange(max_len), y=stats_perfect["min"],
    mode="lines", line=dict(width=0),
    showlegend=False, hoverinfo="skip",
    legendgroup="perfect"
))
fig.add_trace(go.Scatter(
    x=np.arange(max_len), y=stats_perfect["max"],
    mode="lines", line=dict(width=0),
    fill="tonexty", fillcolor=hex_to_rgba(color_perfect, 0.25),
    showlegend=False, hoverinfo="skip",
    legendgroup="perfect"
))

# ---- shaded band : 50 % error ----
fig.add_trace(go.Scatter(
    x=np.arange(max_len), y=stats_error["min"],
    mode="lines", line=dict(width=0),
    showlegend=False, hoverinfo="skip",
    legendgroup="error"
))
fig.add_trace(go.Scatter(
    x=np.arange(max_len), y=stats_error["max"],
    mode="lines", line=dict(width=0),
    fill="tonexty", fillcolor=hex_to_rgba(color_error, 0.25),
    showlegend=False, hoverinfo="skip",
    legendgroup="error"
))

# ---- mean curves ----
fig.add_trace(go.Scatter(
    x=np.arange(max_len), y=stats_perfect["mean"],
    mode="lines",
    name="Perfect information (mean)",
    line=dict(color=color_perfect, width=5),
    legendgroup="perfect"
))
fig.add_trace(go.Scatter(
    x=np.arange(max_len), y=stats_error["mean"],
    mode="lines",
    name="50 % error information (mean)",
    line=dict(color=color_error, width=5),
    legendgroup="error"
))

# ---- safety threshold ----
fig.add_trace(go.Scatter(
    x=[0, max_len - 1], y=[0.95, 0.95],
    mode="lines", name="Safety limit 0.95 p.u.",
    line=dict(color="black", dash="dash"),
    hoverinfo="skip"
))

# ------------------------------------------------------------
# layout: Nature style
# ------------------------------------------------------------
fig.update_layout(
    width=NATURE_CONFIG["width"],
    height=NATURE_CONFIG["height"],
    template="plotly_white",
    title=dict(
        text=f"Average Voltage on Bus 30: 0% vs 50 % topology information loss",
        font=dict(size=NATURE_CONFIG["font_title"])
    ),
    font=dict(size=NATURE_CONFIG["font_base"]),
    legend=dict(
        x=1, y=0.99, xanchor="right", yanchor="top",
        font=dict(size=NATURE_CONFIG["font_legend"])
    ),
    xaxis=dict(
        title="Step",
        titlefont=dict(size=NATURE_CONFIG["font_axis"]),
        tickfont=dict(size=NATURE_CONFIG["font_axis"]),
        range=[0, 18],
    ),
    yaxis=dict(
        title="Voltage (p.u.)",
        titlefont=dict(size=NATURE_CONFIG["font_axis"]),
        tickfont=dict(size=NATURE_CONFIG["font_axis"])
    )
)

fig.show()

# ------------------------------------------------------------
# export high-resolution images
# ------------------------------------------------------------
pio.kaleido.scope.mathjax = None  # speed-up export

output_dir = os.path.join(Config.data_path, "images", "56bus")
os.makedirs(output_dir, exist_ok=True)

pdf_path = os.path.join(output_dir, "topology_loss.pdf")
png_path = os.path.join(output_dir, "topology_loss.png")

fig.write_image(pdf_path,  width=NATURE_CONFIG["width"],
                           height=NATURE_CONFIG["height"],
                           scale=2)   # 2 × 1800 px ≈ 300 dpi
fig.write_image(png_path,  width=NATURE_CONFIG["width"],
                           height=NATURE_CONFIG["height"],
                           scale=2)

average recovery step is:
14.625
15.063013476725034
average reactive power cost is:
54.58733743407025
101.6204096690605
the total cost is:
238.98502216379714
262.97381806591187


C:\Users\wdyao\AppData\Local\Temp\ipykernel_5000\3482050656.py:38: RuntimeWarning:

Mean of empty slice

C:\Users\wdyao\AppData\Local\Temp\ipykernel_5000\3482050656.py:39: RuntimeWarning:

All-NaN slice encountered

C:\Users\wdyao\AppData\Local\Temp\ipykernel_5000\3482050656.py:40: RuntimeWarning:

All-NaN slice encountered



In [ ]:
### test our controller without topology error
voltage_ = []
q_ = []
cost_ = []
success_list_ = []
fail_list_ = []
entire_list_ = []
control_cost_ = []
reward_list_ = []
object_cost_list_ = []

for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.cuda.FloatTensor(topology).unsqueeze(0) * 0.0
    # print(f'Topology for episode {episode}: {topology}')

    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    cost_ = []
    abnormal_stop = False

    for step in range(step_num):
        action = []
        for i in range(agent_num):
            action_agent = agent_policy_net[i](torch.cuda.FloatTensor(state[i].reshape(1,)).unsqueeze(0), topology)
            action_agent = 0.6* action_agent.detach().cpu().numpy()[0]
            action.append(action_agent)

        action = last_action - np.asarray(action)
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            fail_list_.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            fail_list_.append((episode,step))
            abnormal_stop = True
            break
        if done:
            success_list_.append((episode,step))
            logger.success('episode {} stable at {} steps',success_list_[-1][0], success_list_[-1][1])
            break

        voltage_.append(state)

        q_.append(action)

        state = next_state
        
        episode_reward += reward
        
        cost_.append(-reward)
        
        episode_control += LA.norm(action, 2)

    reward_list_.append(episode_reward)
    control_cost_.append(episode_control)
    object_cost_list_.append(np.sum(cost_))

    if (not done) and (abnormal_stop == False):
        entire_list_.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

logger.info('total success epsisode is {}', len(success_list_))
logger.info('total fail episode is {}', len(fail_list_))
logger.info('number of finished at entire episode is {}', len(entire_list_))

success_list_ = np.array(success_list_)
print('average recovery step is:')
print(np.mean(success_list_[:,1]))
print(np.std(success_list_[:,1]))
print('average reactive power cost is:')
print(np.mean(control_cost_))
print(np.std(control_cost_))
print('the total cost is:')
print(np.mean(object_cost_list_))
print(np.std(object_cost_list_))

2025-06-12 11:35:37.658 | SUCCESS  | __main__:<module>:50 - episode 0 stable at 27 steps
2025-06-12 11:35:38.151 | SUCCESS  | __main__:<module>:50 - episode 1 stable at 10 steps
2025-06-12 11:35:39.520 | SUCCESS  | __main__:<module>:50 - episode 2 stable at 35 steps
2025-06-12 11:35:40.370 | SUCCESS  | __main__:<module>:50 - episode 3 stable at 20 steps
2025-06-12 11:35:41.431 | SUCCESS  | __main__:<module>:50 - episode 4 stable at 26 steps
2025-06-12 11:35:43.229 | SUCCESS  | __main__:<module>:50 - episode 5 stable at 44 steps
2025-06-12 11:35:43.605 | SUCCESS  | __main__:<module>:50 - episode 6 stable at 8 steps
2025-06-12 11:35:48.242 | SUCCESS  | __main__:<module>:50 - episode 7 stable at 116 steps
2025-06-12 11:35:51.044 | SUCCESS  | __main__:<module>:50 - episode 8 stable at 73 steps
2025-06-12 11:35:52.543 | SUCCESS  | __main__:<module>:50 - episode 9 stable at 38 steps
2025-06-12 11:35:54.447 | SUCCESS  | __main__:<module>:50 - episode 10 stable at 47 steps
2025-06-12 11:35:55.

average recovery step is:
31.585
28.96588985341206
average reactive power cost is:
120.46901415528887
232.34779806492892
the total cost is:
516.8304113980473
559.5291938315513


### baseline

In [ ]:
### test the base line controller
num_agent = 5
voltage = []
q = []
base_cost = []
base_succ_list = []
base_fail_list = []
base_entire_list = []
base_control_cost = []
base_reward_list = []
base_object_cost = []

for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    base_cost = []
    abnormal_stop = False

    for step in range(step_num):
        state1 = np.asarray(state-env.vmax)         # 就算 +0.001，效果也不如我们的
        state2 = np.asarray(env.vmin-state)
        d_v = (np.maximum(state1, 0)-np.maximum(state2, 0)).reshape((num_agent,1))
        
        action = (last_action - 10*d_v)
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            base_fail_list.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            base_fail_list.append((episode,step))
            abnormal_stop = True
            break
        if done:
            base_succ_list.append((episode,step))
            logger.success('stable at {}',base_succ_list[-1])
            break

        voltage.append(state)

        q.append(action)

        state = next_state
        
        episode_reward += reward
        
        base_cost.append(-reward)
        
        episode_control += LA.norm(action, 2)

    base_control_cost.append(episode_control)
    base_reward_list.append(episode_reward)
    base_object_cost.append(np.sum(base_cost))
    
    if (not done) and (abnormal_stop == False):
        base_entire_list.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

logger.info('total success epsisode is {}', len(base_succ_list))
logger.info('total fail episode is {}', len(base_fail_list))
logger.info('number of finished at entire episode is {}', len(base_entire_list))

2025-08-11 12:44:49.825 | SUCCESS  | __main__:<module>:47 - stable at (0, 49)
2025-08-11 12:44:49.969 | SUCCESS  | __main__:<module>:47 - stable at (1, 11)
2025-08-11 12:44:50.751 | SUCCESS  | __main__:<module>:47 - stable at (2, 77)
2025-08-11 12:44:51.175 | SUCCESS  | __main__:<module>:47 - stable at (3, 39)
2025-08-11 12:44:51.598 | SUCCESS  | __main__:<module>:47 - stable at (4, 39)
2025-08-11 12:44:52.566 | SUCCESS  | __main__:<module>:47 - stable at (5, 94)
2025-08-11 12:44:52.660 | SUCCESS  | __main__:<module>:47 - stable at (6, 6)
2025-08-11 12:44:53.603 | SUCCESS  | __main__:<module>:47 - stable at (7, 91)
2025-08-11 12:44:54.816 | SUCCESS  | __main__:<module>:47 - stable at (8, 119)
2025-08-11 12:44:55.056 | SUCCESS  | __main__:<module>:47 - stable at (9, 19)
2025-08-11 12:44:55.814 | SUCCESS  | __main__:<module>:47 - stable at (10, 74)
2025-08-11 12:44:56.177 | SUCCESS  | __main__:<module>:47 - stable at (11, 32)
2025-08-11 12:44:56.400 | SUCCESS  | __main__:<module>:47 - st

In [ ]:
base_succ_list = np.array(base_succ_list)
print('average recovery step is:')
print(np.mean(base_succ_list[:,1]))
print(np.std(base_succ_list[:,1]))
print('average reactive power cost is:')
print(np.mean(base_control_cost))
print(np.std(base_control_cost))
print('the total cost is:')
print(base_object_cost)
print(np.mean(base_object_cost))
print(np.std(base_object_cost))


average recovery step is:
39.05
30.58737484649508
average reactive power cost is:
179.5313379054742
312.6940396299014
the total cost is:
[np.float64(620.6944965909221), np.float64(119.69188443265773), np.float64(1067.8934783016796), np.float64(499.74733102956503), np.float64(412.30634140889435), np.float64(1499.4581355391151), np.float64(71.41584134224925), np.float64(1778.0421392801152), np.float64(1888.0331570204523), np.float64(226.12450486156106), np.float64(988.7679092840652), np.float64(483.4618967529997), np.float64(247.7245112024075), np.float64(633.9812426540907), np.float64(128.4622973377111), np.float64(143.2096969188107), np.float64(846.3717202865423), np.float64(1814.1314844680944), np.float64(318.916353842823), np.float64(880.1665048419944), np.float64(0.0), np.float64(144.47205556878913), np.float64(152.9078236547165), np.float64(544.0551924480042), np.float64(292.79686742921064), np.float64(136.9804904450139), np.float64(535.5500725343883), np.float64(523.749034137109),

### Safe DDPG

In [ ]:
### test the safe policy net
num_agent = 5
safe_voltage = []
safe_q = []
safe_cost = []
safe_succ_list = []
safe_fail_list = []
safe_entire_list = []
safe_contorl_cost = []
safe_reward_list = []
safe_object_cost = []

for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    safe_cost = []
    abnormal_stop = False

    for step in range(step_num):
        action = []
        for i in range(num_agent):
            action_agent = safe_agent_net[i].get_action(torch.cuda.FloatTensor([state[i]]).float().reshape(1,1))
            action.append(action_agent)
        
        action = last_action - 5*np.asarray(action).reshape((num_agent, 1))
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            safe_fail_list.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            safe_fail_list.append((episode,step))
            abnormal_stop = True
            break
        if done:
            safe_succ_list.append((episode,step))
            logger.success('stable at {}',safe_succ_list[-1])
            break
        safe_voltage.append(state)

        safe_q.append(action)

        state = next_state
        
        episode_reward += reward
        
        safe_cost.append(-reward)
        
        episode_control += LA.norm(action, 2)

    safe_contorl_cost.append(episode_control)
    safe_reward_list.append(episode_reward)
    safe_object_cost.append(np.sum(safe_cost))

    if (not done) and (abnormal_stop == False):
        safe_entire_list.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

logger.info('total success epsisode is {}', len(safe_succ_list))
logger.info('total fail episode is {}', len(safe_fail_list))
logger.info('number of finished at entire episode is {}', len(safe_entire_list))


2025-08-11 12:47:05.679 | SUCCESS  | __main__:<module>:48 - stable at (0, 32)
2025-08-11 12:47:05.850 | SUCCESS  | __main__:<module>:48 - stable at (1, 5)
2025-08-11 12:47:07.090 | SUCCESS  | __main__:<module>:48 - stable at (2, 53)
2025-08-11 12:47:07.682 | SUCCESS  | __main__:<module>:48 - stable at (3, 23)
2025-08-11 12:47:08.198 | SUCCESS  | __main__:<module>:48 - stable at (4, 21)
2025-08-11 12:47:09.910 | SUCCESS  | __main__:<module>:48 - stable at (5, 66)
2025-08-11 12:47:09.968 | SUCCESS  | __main__:<module>:48 - stable at (6, 0)
2025-08-11 12:47:11.425 | SUCCESS  | __main__:<module>:48 - stable at (7, 63)
2025-08-11 12:47:13.559 | SUCCESS  | __main__:<module>:48 - stable at (8, 83)
2025-08-11 12:47:13.842 | SUCCESS  | __main__:<module>:48 - stable at (9, 10)
2025-08-11 12:47:15.249 | SUCCESS  | __main__:<module>:48 - stable at (10, 51)
2025-08-11 12:47:15.753 | SUCCESS  | __main__:<module>:48 - stable at (11, 18)
2025-08-11 12:47:16.023 | SUCCESS  | __main__:<module>:48 - stab

In [ ]:
safe_succ_list = np.array(safe_succ_list)
print('average recovery step is:')
print(np.mean(safe_succ_list[:,1]))
print(np.std(safe_succ_list[:,1]))
print('average reactive power cost is:')
print(np.mean(safe_contorl_cost))
print(np.std(safe_contorl_cost))
print('the total cost is:')
print(safe_object_cost)
print(np.mean(safe_object_cost))
print(np.std(safe_object_cost))

average recovery step is:
23.88
22.89073174889785
average reactive power cost is:
118.19412608463817
217.3431220496375
the total cost is:
[np.float64(403.27794474390487), np.float64(55.32541483771597), np.float64(727.1358443366639), np.float64(294.4688489239965), np.float64(202.61153268247136), np.float64(1044.1712706707374), np.float64(0.0), np.float64(1231.4249116000344), np.float64(1315.220563460281), np.float64(118.9368137269885), np.float64(670.7839783985423), np.float64(274.1472844934442), np.float64(127.54566447653293), np.float64(423.3392388423217), np.float64(58.65874583048778), np.float64(0.0), np.float64(580.6666348148702), np.float64(1253.1081216777854), np.float64(184.43242807283775), np.float64(585.2496031743691), np.float64(0.0), np.float64(63.721663747790004), np.float64(64.95427916838742), np.float64(316.49606392562606), np.float64(150.98023285824613), np.float64(65.17465099812792), np.float64(332.18825766725683), np.float64(160.70080637364015), np.float64(194.89533822

In [ ]:
import plotly.graph_objects as go
import numpy as np
from plotly.subplots import make_subplots

# 从原始数据计算统计值
methods = ['Linear', 'Safe-DDPG', 'RLC-FT']
metrics = ['Voltage Recovery Time', 'Reactive Power', 'Total Objective Cost']

# 提取数据并计算统计值
data = {
    'Voltage Recovery Time': {
        'means': [
            np.mean(base_succ_list[:,1]),      # Linear
            np.mean(safe_succ_list[:,1]),      # Safe-DDPG
            np.mean(success_list[:,1])         # RLC-FT
        ],
        'stds': [
            np.std(base_succ_list[:,1]),       # Linear
            np.std(safe_succ_list[:,1]),       # Safe-DDPG
            np.std(success_list[:,1])          # RLC-FT
        ],
        'unit': '(Steps)'
    },
    'Reactive Power': {
        'means': [
            np.mean(base_control_cost),        # Linear
            np.mean(safe_contorl_cost),        # Safe-DDPG (注意原始变量名的拼写)
            np.mean(control_cost)              # RLC-FT
        ],
        'stds': [
            np.std(base_control_cost),         # Linear
            np.std(safe_contorl_cost),         # Safe-DDPG
            np.std(control_cost)               # RLC-FT
        ],
        'unit': '(MVar)'
    },
    'Total Objective Cost': {
        'means': [
            np.mean(base_object_cost),         # Linear
            np.mean(safe_object_cost),         # Safe-DDPG
            np.mean(object_cost)               # RLC-FT
        ],
        'stds': [
            np.std(base_object_cost),          # Linear
            np.std(safe_object_cost),          # Safe-DDPG
            np.std(object_cost)                # RLC-FT
        ],
        'unit': ''
    }
}

# 打印计算结果用于验证
print("Calculated Statistics:")
for metric in metrics:
    print(f"\n{metric}:")
    for i, method in enumerate(methods):
        print(f"  {method}: {data[metric]['means'][i]:.2f} ± {data[metric]['stds'][i]:.2f}")

# Nature风格的颜色
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

# 创建子图
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f"{metric}<br> {data[metric]['unit']}" for metric in metrics],
    horizontal_spacing=0.15
)

# 为每个指标创建分组柱状图
for i, metric in enumerate(metrics):
    means = data[metric]['means']
    stds = data[metric]['stds']
    
    for j, method in enumerate(methods):
        fig.add_trace(
            go.Bar(
                x=[method],
                y=[means[j]],
                error_y=dict(
                    type='data',
                    array=[stds[j]],
                    visible=True,
                    thickness=2,
                    width=3
                ),
                name=method,
                marker_color=colors[j],
                marker_line=dict(width=0.5, color='black'),
                showlegend=(i == 0),
                width=0.6
            ),
            row=1, col=i+1
        )
        
        # 添加数值标签在误差线上方
        fig.add_annotation(
            x=method,
            y=means[j] + max(means) * 0.1,  # 在误差线上方
            xshift=18,  # 向右偏移数值标签，单位为像素，可根据需要调整
            text=f"{means[j]:.1f}",
            showarrow=False,
            font=dict(size=10, color='black'),
            row=1, col=i+1
        )


# 更新布局
fig.update_layout(
    width=900,
    height=450,
    font=dict(
        family="Arial, sans-serif",
        size=12,
        color="black"
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(
        x=1.02,
        y=1,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255,255,255,0)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=11)
    ),
    title=dict(
        text="Performance Comparison of Different Controllers in 56-bus System",
        x=0.5,
        xanchor='center',
        font=dict(size=14, color='black')
    ),
    margin=dict(l=60, r=120, t=100, b=60)  # 增加top margin以容纳数值标签
)

# 更新轴样式
for i in range(1, 4):
    fig.update_xaxes(
        row=1, col=i,
        showgrid=False,
        showline=True,
        linewidth=1,
        linecolor='black',
        tickfont=dict(size=10)
    )
    
    fig.update_yaxes(
        row=1, col=i,
        showgrid=True,
        gridwidth=0.5,
        gridcolor='lightgray',
        showline=True,
        linewidth=1,
        linecolor='black',
        tickfont=dict(size=10)
    )

# 更新子图标题格式
for annotation in fig['layout']['annotations']:
    annotation['font'] = dict(size=12, color='black')

# 显示图形
fig.show()

# 计算性能改善百分比
print("\n" + "="*50)
print("Performance Improvement of RLC-FT vs Linear:")
recovery_improve = (data['Voltage Recovery Time']['means'][0] - data['Voltage Recovery Time']['means'][2]) / data['Voltage Recovery Time']['means'][0] * 100
power_improve = (data['Reactive Power']['means'][0] - data['Reactive Power']['means'][2]) / data['Reactive Power']['means'][0] * 100
cost_improve = (data['Total Objective Cost']['means'][0] - data['Total Objective Cost']['means'][2]) / data['Total Objective Cost']['means'][0] * 100

print(f"• Voltage Recovery: {recovery_improve:.1f}% faster")
print(f"• Reactive Power: {power_improve:.1f}% reduction")
print(f"• Total Cost: {cost_improve:.1f}% reduction")

# 保存高质量图片（可选）
# fig.write_image("performance_comparison.png", width=1000, height=400, scale=3)
# fig.write_image("performance_comparison.pdf", width=1000, height=400)

Calculated Statistics:

Voltage Recovery Time:
  Linear: 39.05 ± 30.59
  Safe-DDPG: 23.88 ± 22.89
  RLC-FT: 8.04 ± 8.60

Reactive Power:
  Linear: 179.53 ± 312.69
  Safe-DDPG: 118.19 ± 217.34
  RLC-FT: 33.73 ± 66.65

Total Objective Cost:
  Linear: 552.85 ± 548.85
  Safe-DDPG: 340.27 ± 398.38
  RLC-FT: 137.33 ± 159.70



Performance Improvement of RLC-FT vs Linear:
• Voltage Recovery: 79.4% faster
• Reactive Power: 81.2% reduction
• Total Cost: 75.2% reduction


In [ ]:
import plotly.graph_objects as go
import numpy as np
from plotly.subplots import make_subplots

# Calculate statistics from raw data
methods = ['Linear', 'Safe-DDPG', 'RLC-FT']
metrics = ['Voltage Recovery Time', 'Reactive Power', 'Total Objective Cost']

# Extract data and calculate statistical values
data = {
    'Voltage Recovery Time': {
        'means': [
            np.mean(base_succ_list[:,1]),      # Linear
            np.mean(safe_succ_list[:,1]),      # Safe-DDPG
            np.mean(success_list[:,1])         # RLC-FT
        ],
        'stds': [
            np.std(base_succ_list[:,1]),       # Linear
            np.std(safe_succ_list[:,1]),       # Safe-DDPG
            np.std(success_list[:,1])          # RLC-FT
        ],
        'unit': 'Steps'
    },
    'Reactive Power': {
        'means': [
            np.mean(base_control_cost),        # Linear
            np.mean(safe_contorl_cost),        # Safe-DDPG (note original variable name spelling)
            np.mean(control_cost)              # RLC-FT
        ],
        'stds': [
            np.std(base_control_cost),         # Linear
            np.std(safe_contorl_cost),         # Safe-DDPG
            np.std(control_cost)               # RLC-FT
        ],
        'unit': 'MVar'
    },
    'Total Objective Cost': {
        'means': [
            np.mean(base_object_cost),         # Linear
            np.mean(safe_object_cost),         # Safe-DDPG
            np.mean(object_cost)               # RLC-FT
        ],
        'stds': [
            np.std(base_object_cost),          # Linear
            np.std(safe_object_cost),          # Safe-DDPG
            np.std(object_cost)                # RLC-FT
        ],
        'unit': ''
    }
}

# Print calculated results for verification (including truncation info)
print("Calculated Statistics:")
truncation_info = []
for metric in metrics:
    print(f"\n{metric}:")
    for i, method in enumerate(methods):
        mean_val = data[metric]['means'][i]
        std_val = data[metric]['stds'][i]
        lower_bound = mean_val - std_val
        
        if lower_bound < 0:
            truncation_info.append(f"{metric} - {method}")
            print(f"  {method}: {mean_val:.2f} ± {std_val:.2f} (truncated at 0)")
        else:
            print(f"  {method}: {mean_val:.2f} ± {std_val:.2f}")

# Nature-style colors
colors = {
    'Linear':  '#5AA1D6',  # lighter blue from #0072B2
    'Safe-DDPG': '#E7904B',# lighter orange from #D55E00
    'RLC-FT':  '#53C19E'   # lighter green from #009E73
}
colors_edge = {
    'Linear':  '#2F6E9D',
    'Safe-DDPG': '#B3611E',
    'RLC-FT':  '#2D8C73'
}
err_color = 'rgba(0,0,0,0.55)'  # unified error bar color

# Create subplots
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[metric for metric in metrics],  # Only metric names, no units
    horizontal_spacing=0.15
)

# Create grouped bar charts for each metric
for i, metric in enumerate(metrics):
    means = data[metric]['means']
    stds = data[metric]['stds']
    
    for j, method in enumerate(methods):
        mean_val = means[j]
        std_val = stds[j]
        
        # Check if truncation is needed (handle negative values)
        is_truncated = (mean_val - std_val) < 0
        
        # Calculate error bar lengths
        if is_truncated:
            # Truncation handling: lower error bar only extends to 0
            error_minus = mean_val  # distance from mean to 0
            error_plus = std_val    # keep original upper error bar
        else:
            # Normal case: symmetric error bars
            error_minus = std_val
            error_plus = std_val
        
        bar_color = colors[method]
        edge_color = colors_edge[method]

        fig.add_trace(
            go.Bar(
                x=[method],
                y=[mean_val],
                error_y=dict(
                    type='data',
                    symmetric=False,
                    array=[error_plus],
                    arrayminus=[error_minus],
                    visible=True,
                    thickness=2.5,
                    width=5,
                    color=err_color
                ),
                name=method,
                marker_color=bar_color,
                marker_line=dict(width=1.6, color=edge_color),
                showlegend=(i == 0),
                width=0.6
            ),
            row=1, col=i+1

        )
        
        # Add value labels above error bars
        fig.add_annotation(
            x=method,
            y=mean_val + max(means) * 0.1,  # above error bars
            xshift=30,  # shift value labels to the right (in pixels, adjustable)
            text=f"{mean_val:.1f}",
            showarrow=False,
            font=dict(size=16, color='black'),
            row=1, col=i+1
        )
        
        # Add truncation marker at bottom if truncated
        if is_truncated:
            fig.add_annotation(
                x=method,
                y=max(means) * 0.02,  # near bottom
                text="⊥",  # truncation symbol
                showarrow=False,
                font=dict(size=16, color='red'),
                row=1, col=i+1
            )

# Update layout (removed title)
fig.update_layout(
    width=1200,
    height=500,
    font=dict(family="Arial, sans-serif", size=20, color="black"),
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(
        x=1.02, y=1,
        xanchor='left', yanchor='top',
        bgcolor='rgba(255,255,255,0.6)',
        bordercolor='rgba(0,0,0,0)',
        font=dict(size=18)
    ),
    margin=dict(l=70, r=140, t=80, b=70)  # reduced top margin since no title
)

# Update axis styles with units on y-axes
y_axis_titles = [
    data['Voltage Recovery Time']['unit'],
    data['Reactive Power']['unit'], 
    data['Total Objective Cost']['unit']
]

for i in range(1, 4):
    fig.update_xaxes(
        row=1, col=i,
        showgrid=False,
        showline=True,
        linewidth=1.5,
        linecolor='black',
        tickfont=dict(size=18)
    )
    
    # Set y-axis title with unit
    y_title = y_axis_titles[i-1] if y_axis_titles[i-1] else ""
    
    fig.update_yaxes(
        row=1, col=i,
        title=dict(
            text=y_title,
            font=dict(size=18, color='black')
        ),
        showgrid=True,
        gridwidth=0.5,
        gridcolor='lightgray',
        showline=True,
        linewidth=1.5,
        linecolor='black',
        tickfont=dict(size=18),
        rangemode='tozero'  # ensure y-axis starts from 0
    )

# Update subplot title formatting (without units now)
for annotation in fig['layout']['annotations']:
    annotation['font'] = dict(size=20, color='black')

# Display figure
fig.show()

# Display truncation information
if truncation_info:
    print(f"\n⊥ Truncation Applied (error bars cut at zero for):")
    for info in truncation_info:
        print(f"  - {info}")
    print("Note: Red ⊥ symbols indicate where error bars were truncated at zero")

# Data integrity check
print("\n" + "="*50)
print("Data Quality Summary:")
for metric in metrics:
    means = data[metric]['means']
    stds = data[metric]['stds']
    
    print(f"\n{metric}:")
    for j, method in enumerate(methods):
        cv = (stds[j] / means[j]) * 100  # coefficient of variation
        truncated = "Yes" if (means[j] - stds[j]) < 0 else "No"
        print(f"  {method}: CV={cv:.1f}%, Truncated={truncated}")

# Save high-quality images (optional)
fig.write_image(os.path.join(Config.data_path, "images","56bus", "performance_comparison.png"), width=1200, height=500, scale=2)
fig.write_image(os.path.join(Config.data_path, "images","56bus", "performance_comparison.pdf"), width=1200, height=500)

Calculated Statistics:

Voltage Recovery Time:
  Linear: 39.05 ± 30.59
  Safe-DDPG: 23.88 ± 22.89
  RLC-FT: 8.04 ± 8.60 (truncated at 0)

Reactive Power:
  Linear: 179.53 ± 312.69 (truncated at 0)
  Safe-DDPG: 118.19 ± 217.34 (truncated at 0)
  RLC-FT: 33.73 ± 66.65 (truncated at 0)

Total Objective Cost:
  Linear: 552.85 ± 548.85
  Safe-DDPG: 340.27 ± 398.38 (truncated at 0)
  RLC-FT: 137.33 ± 159.70 (truncated at 0)



⊥ Truncation Applied (error bars cut at zero for):
  - Voltage Recovery Time - RLC-FT
  - Reactive Power - Linear
  - Reactive Power - Safe-DDPG
  - Reactive Power - RLC-FT
  - Total Objective Cost - Safe-DDPG
  - Total Objective Cost - RLC-FT
Note: Red ⊥ symbols indicate where error bars were truncated at zero

Performance Improvement of RLC-FT vs Linear:
• Voltage Recovery: 79.4% faster
• Reactive Power: 81.2% reduction
• Total Cost: 75.2% reduction

Data Quality Summary:

Voltage Recovery Time:
  Linear: CV=78.3%, Truncated=No
  Safe-DDPG: CV=95.9%, Truncated=No
  RLC-FT: CV=106.9%, Truncated=Yes

Reactive Power:
  Linear: CV=174.2%, Truncated=Yes
  Safe-DDPG: CV=183.9%, Truncated=Yes
  RLC-FT: CV=197.6%, Truncated=Yes

Total Objective Cost:
  Linear: CV=99.3%, Truncated=No
  Safe-DDPG: CV=117.1%, Truncated=Yes
  RLC-FT: CV=116.3%, Truncated=Yes


In [20]:
import plotly.graph_objects as go
import numpy as np
from plotly.subplots import make_subplots

# Calculate statistics from raw data
methods = ['Linear', 'Safe-DDPG', 'RLC-FT']
metrics = ['Voltage Recovery Time', 'Reactive Power', 'Total Objective Cost']

# Extract data and calculate statistical values
data = {
    'Voltage Recovery Time': {
        'means': [
            39.05,      # Linear
            23.88,      # Safe-DDPG
            8.04         # RLC-FT
        ],
        'stds': [
            30.59,       # Linear
            22.89,       # Safe-DDPG
            8.60           # RLC-FT
        ],
        'unit': 'Steps'
    },
    'Reactive Power': {
        'means': [
            179.53,        # Linear
            118.19,        # Safe-DDPG (note original variable name spelling)
            33.73              # RLC-FT
        ],
        'stds': [
            312.69,         # Linear
            217.34,         # Safe-DDPG
            66.65               # RLC-FT
        ],
        'unit': 'MVar'
    },
    'Total Objective Cost': {
        'means': [
            552.85,         # Linear
            340.27,         # Safe-DDPG
            137.33               # RLC-FT
        ],
        'stds': [
            548.85,          # Linear
            398.38,          # Safe-DDPG
            159.70                # RLC-FT
        ],
        'unit': ''
    }
}

# data = {
#     'Voltage Recovery Time': {
#         'means': [
#             106.38,      # Linear
#             51.59,      # Safe-DDPG
#             17.99         # RLC-FT
#         ],
#         'stds': [
#             38.84,       # Linear
#             23.62,       # Safe-DDPG
#             12.10          # RLC-FT
#         ],
#         'unit': 'Steps'
#     },
#     'Reactive Power': {
#         'means': [
#             2034.35,        # Linear
#             1087.41,        # Safe-DDPG (note original variable name spelling)
#             271.49              # RLC-FT
#         ],
#         'stds': [
#             1696.93,         # Linear
#             1017.19,         # Safe-DDPG
#             287.73               # RLC-FT
#         ],
#         'unit': 'MVar'
#     },
#     'Total Objective Cost': {
#         'means': [
#             320150.54,         # Linear
#             177025.43,         # Safe-DDPG
#             50326.50               # RLC-FT
#         ],
#         'stds': [
#             183398.67,          # Linear
#             101214.86,          # Safe-DDPG
#             28921.66                # RLC-FT
#         ],
#         'unit': ''
#     }
# }

# Print calculated results for verification (including truncation info)
print("Calculated Statistics:")
truncation_info = []
for metric in metrics:
    print(f"\n{metric}:")
    for i, method in enumerate(methods):
        mean_val = data[metric]['means'][i]
        std_val = data[metric]['stds'][i]
        lower_bound = mean_val - std_val
        
        if lower_bound < 0:
            truncation_info.append(f"{metric} - {method}")
            print(f"  {method}: {mean_val:.2f} ± {std_val:.2f} (truncated at 0)")
        else:
            print(f"  {method}: {mean_val:.2f} ± {std_val:.2f}")

# Nature-style colors
# colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
# colors = ['#0072B2', '#D55E00', '#009E73']
# Softer Okabe–Ito palette (lighter fills)
colors = {
    'Linear':  '#5AA1D6',  # lighter blue from #0072B2
    'Safe-DDPG': '#E7904B',# lighter orange from #D55E00
    'RLC-FT':  '#53C19E'   # lighter green from #009E73
}
colors_edge = {
    'Linear':  '#2F6E9D',
    'Safe-DDPG': '#B3611E',
    'RLC-FT':  '#2D8C73'
}
err_color = 'rgba(0,0,0,0.55)'  # unified error bar color

# Create subplots
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[metric for metric in metrics],  # Only metric names, no units
    horizontal_spacing=0.15
)

# Create grouped bar charts for each metric
for i, metric in enumerate(metrics):
    means = data[metric]['means']
    stds = data[metric]['stds']
    
    for j, method in enumerate(methods):
        mean_val = means[j]
        std_val = stds[j]
        
        # Check if truncation is needed (handle negative values)
        is_truncated = (mean_val - std_val) < 0
        
        # Calculate error bar lengths
        if is_truncated:
            # Truncation handling: lower error bar only extends to 0
            error_minus = mean_val  # distance from mean to 0
            error_plus = std_val    # keep original upper error bar
        else:
            # Normal case: symmetric error bars
            error_minus = std_val
            error_plus = std_val
        
        bar_color = colors[method]
        edge_color = colors_edge[method]

        fig.add_trace(
            go.Bar(
                x=[method],
                y=[mean_val],
                error_y=dict(
                    type='data',
                    symmetric=False,
                    array=[error_plus],
                    arrayminus=[error_minus],
                    visible=True,
                    thickness=2.5,
                    width=5,
                    color=err_color
                ),
                name=method,
                marker_color=bar_color,
                marker_line=dict(width=1.6, color=edge_color),
                showlegend=(i == 0),
                width=0.6
            ),
            row=1, col=i+1

        )
        
        # Add value labels above error bars
        fig.add_annotation(
            x=method,
            y=mean_val + max(means) * 0.1,  # above error bars
            xshift=30,  # shift value labels to the right (in pixels, adjustable)
            text=f"{mean_val:.1f}",
            showarrow=False,
            font=dict(size=16, color='black'),
            row=1, col=i+1
        )

        # Add value labels above error bars
        # if i == 0:  #
        #     fig.add_annotation(
        #         x=method,
        #         y=mean_val + max(means) * 0.05,  # above error bars
        #         xshift=25,  # shift value labels to the right (in pixels, adjustable)
        #         text=f"{mean_val:.1f}",
        #         showarrow=False,
        #         font=dict(size=14, color='black'),
        #         row=1, col=i+1
        #     )

        # if i == 1:  #
        #     fig.add_annotation(
        #         x=method,
        #         y=mean_val + max(means) * 0.08,  # above error bars
        #         xshift=32,  # shift value labels to the right (in pixels, adjustable)
        #         text=f"{mean_val:.1f}",
        #         showarrow=False,
        #         font=dict(size=14, color='black'),
        #         row=1, col=i+1
        #     )

        # if i == 2:  #
        #     fig.add_annotation(
        #         x=method,
        #         y=mean_val + max(means) * 0.08,  # above error bars
        #         xshift=45,  # shift value labels to the right (in pixels, adjustable)
        #         text=f"{mean_val:.1f}",
        #         showarrow=False,
        #         font=dict(size=14, color='black'),
        #         row=1, col=i+1
        #     )
        
        # Add truncation marker at bottom if truncated
        if is_truncated:
            fig.add_annotation(
                x=method,
                y=max(means) * 0.02,  # near bottom
                text="⊥",  # truncation symbol
                showarrow=False,
                font=dict(size=16, color='red'),
                row=1, col=i+1
            )

# Update layout (removed title)
fig.update_layout(
    width=1200,
    height=500,
    font=dict(family="Arial, sans-serif", size=20, color="black"),
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(
        x=1.02, y=1,
        xanchor='left', yanchor='top',
        bgcolor='rgba(255,255,255,0.6)',
        bordercolor='rgba(0,0,0,0)',
        font=dict(size=18)
    ),
    margin=dict(l=70, r=140, t=80, b=70)  # reduced top margin since no title
)

# Update axis styles with units on y-axes
y_axis_titles = [
    data['Voltage Recovery Time']['unit'],
    data['Reactive Power']['unit'], 
    data['Total Objective Cost']['unit']
]

for i in range(1, 4):
    fig.update_xaxes(
        row=1, col=i,
        showgrid=False,
        showline=True,
        linewidth=1.5,
        linecolor='black',
        tickfont=dict(size=18)
    )
    
    # Set y-axis title with unit
    y_title = y_axis_titles[i-1] if y_axis_titles[i-1] else ""
    
    fig.update_yaxes(
        row=1, col=i,
        title=dict(
            text=y_title,
            font=dict(size=18, color='black')
        ),
        showgrid=True,
        gridwidth=0.5,
        gridcolor='lightgray',
        showline=True,
        linewidth=1.5,
        linecolor='black',
        tickfont=dict(size=18),
        rangemode='tozero'  # ensure y-axis starts from 0
    )

# Update subplot title formatting (without units now)
for annotation in fig['layout']['annotations']:
    annotation['font'] = dict(size=20, color='black')

# Display figure
fig.show()

# Display truncation information
if truncation_info:
    print(f"\n⊥ Truncation Applied (error bars cut at zero for):")
    for info in truncation_info:
        print(f"  - {info}")
    print("Note: Red ⊥ symbols indicate where error bars were truncated at zero")

# Data integrity check
print("\n" + "="*50)
print("Data Quality Summary:")
for metric in metrics:
    means = data[metric]['means']
    stds = data[metric]['stds']
    
    print(f"\n{metric}:")
    for j, method in enumerate(methods):
        cv = (stds[j] / means[j]) * 100  # coefficient of variation
        truncated = "Yes" if (means[j] - stds[j]) < 0 else "No"
        print(f"  {method}: CV={cv:.1f}%, Truncated={truncated}")

# Save high-quality images (optional)
# fig.write_image(os.path.join(Config.data_path, "images","56bus", "performance_comparison_123.png"), width=1200, height=500, scale=2)
# fig.write_image(os.path.join(Config.data_path, "images","56bus", "performance_comparison_123.pdf"), width=1200, height=500)

fig.write_image(os.path.join(Config.data_path, "images","56bus", "performance_comparison.png"), width=1200, height=500, scale=2)
fig.write_image(os.path.join(Config.data_path, "images","56bus", "performance_comparison.pdf"), width=1200, height=500)

Calculated Statistics:

Voltage Recovery Time:
  Linear: 39.05 ± 30.59
  Safe-DDPG: 23.88 ± 22.89
  RLC-FT: 8.04 ± 8.60 (truncated at 0)

Reactive Power:
  Linear: 179.53 ± 312.69 (truncated at 0)
  Safe-DDPG: 118.19 ± 217.34 (truncated at 0)
  RLC-FT: 33.73 ± 66.65 (truncated at 0)

Total Objective Cost:
  Linear: 552.85 ± 548.85
  Safe-DDPG: 340.27 ± 398.38 (truncated at 0)
  RLC-FT: 137.33 ± 159.70 (truncated at 0)



⊥ Truncation Applied (error bars cut at zero for):
  - Voltage Recovery Time - RLC-FT
  - Reactive Power - Linear
  - Reactive Power - Safe-DDPG
  - Reactive Power - RLC-FT
  - Total Objective Cost - Safe-DDPG
  - Total Objective Cost - RLC-FT
Note: Red ⊥ symbols indicate where error bars were truncated at zero

Data Quality Summary:

Voltage Recovery Time:
  Linear: CV=78.3%, Truncated=No
  Safe-DDPG: CV=95.9%, Truncated=No
  RLC-FT: CV=107.0%, Truncated=Yes

Reactive Power:
  Linear: CV=174.2%, Truncated=Yes
  Safe-DDPG: CV=183.9%, Truncated=Yes
  RLC-FT: CV=197.6%, Truncated=Yes

Total Objective Cost:
  Linear: CV=99.3%, Truncated=No
  Safe-DDPG: CV=117.1%, Truncated=Yes
  RLC-FT: CV=116.3%, Truncated=Yes
